# Parser Inference FastAPI + ngrok on 

In [1]:
!pip install -q fastapi "uvicorn[standard]" pyngrok transformers peft accelerate bitsandbytes safetensors pydantic jsonschema

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import json
import time
import zipfile
import tempfile
from pathlib import Path
from typing import Any
from collections import defaultdict
import threading
import re
import unicodedata

import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
import uvicorn
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from jsonschema import Draft7Validator

In [3]:
BASE_MODEL_ID = os.getenv("PARSER_BASE_MODEL_ID", "Qwen/Qwen3-4B-Instruct-2507")

ARTIFACT_ZIP_PATH = os.getenv(
    "PARSER_ARTIFACT_ZIP",
    "/kaggle/input/government-parser-qwen3-4b-lora-artifact/government_parser_qwen3_4b_lora_artifact.zip",
)

ADAPTER_PATH = os.getenv(
    "PARSER_ADAPTER_PATH",
    "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter",
)

CONFIG_DIR = Path(os.getenv(
    "PARSER_CONFIG_DIR",
    "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs",
))

EXTRACT_DIR = os.getenv("PARSER_EXTRACT_DIR", "/kaggle/working/government_parser_artifact")
PORT = int(os.getenv("PARSER_SERVICE_PORT", "8000"))

MAX_NEW_TOKENS = int(os.getenv("PARSER_MAX_NEW_TOKENS", "512"))
LOAD_IN_4BIT = os.getenv("PARSER_LOAD_IN_4BIT", "true").lower() == "true"
SERVICE_VERSION = "parser_model_service_v0.3.1"
INFERENCE_MODE = "v3_1_training_prompt_post_guardrails_runtime"


def get_ngrok_token() -> str | None:
    token = os.getenv("NGROK_AUTHTOKEN")
    if token:
        return token

    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        token = user_secrets.get_secret("NGROK_AUTHTOKEN")
        if token:
            return token
    except Exception as exc:
        print("Could not read NGROK_AUTHTOKEN from Kaggle Secrets:", type(exc).__name__)

    return None


NGROK_AUTHTOKEN = get_ngrok_token()

config_summary = {
    "BASE_MODEL_ID": BASE_MODEL_ID,
    "ARTIFACT_ZIP_PATH": ARTIFACT_ZIP_PATH,
    "ADAPTER_PATH": ADAPTER_PATH,
    "CONFIG_DIR": str(CONFIG_DIR),
    "EXTRACT_DIR": EXTRACT_DIR,
    "PORT": PORT,
    "NGROK_AUTHTOKEN_SET": bool(NGROK_AUTHTOKEN),
    "MAX_NEW_TOKENS": MAX_NEW_TOKENS,
    "LOAD_IN_4BIT": LOAD_IN_4BIT,
    "SERVICE_VERSION": SERVICE_VERSION,
    "INFERENCE_MODE": INFERENCE_MODE,
}

print(json.dumps(config_summary, indent=2, ensure_ascii=False))
print("CONFIG_DIR exists:", CONFIG_DIR.exists())

{
  "BASE_MODEL_ID": "Qwen/Qwen3-4B-Instruct-2507",
  "ARTIFACT_ZIP_PATH": "/kaggle/input/government-parser-qwen3-4b-lora-artifact/government_parser_qwen3_4b_lora_artifact.zip",
  "ADAPTER_PATH": "/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter",
  "EXTRACT_DIR": "/kaggle/working/government_parser_artifact",
  "PORT": 8000,
  "NGROK_AUTHTOKEN_SET": true,
  "MAX_NEW_TOKENS": 512,
  "TEMPERATURE": 0.0,
  "TOP_P": 0.95,
  "LOAD_IN_4BIT": true,
  "SERVICE_VERSION": "parser_model_service_v0.1.0"
}
CONFIG_DIR: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs
CONFIG_DIR exists: True


In [4]:
def _find_adapter_dir(root: Path) -> Path | None:
    direct_candidates = [root, root / "final_adapter"]
    for candidate in direct_candidates:
        if candidate.exists() and (candidate / "adapter_config.json").exists():
            return candidate

    for config_path in root.rglob("adapter_config.json"):
        return config_path.parent

    return None


def prepare_adapter_path() -> str:
    if ADAPTER_PATH:
        adapter_path = Path(ADAPTER_PATH).expanduser().resolve()
        if adapter_path.exists() and (adapter_path / "adapter_config.json").exists():
            print(f"Using adapter folder from PARSER_ADAPTER_PATH: {adapter_path}")
            return str(adapter_path)
        raise FileNotFoundError(
            f"PARSER_ADAPTER_PATH is set but adapter_config.json was not found: {adapter_path}"
        )

    artifact_zip_path = Path(ARTIFACT_ZIP_PATH).expanduser()
    if artifact_zip_path.exists():
        extract_dir = Path(EXTRACT_DIR)
        extract_dir.mkdir(parents=True, exist_ok=True)

        print(f"Extracting adapter artifact zip: {artifact_zip_path}")
        with zipfile.ZipFile(artifact_zip_path, "r") as artifact_zip:
            artifact_zip.extractall(extract_dir)

        adapter_dir = _find_adapter_dir(extract_dir)
        if adapter_dir is not None:
            print(f"Using extracted adapter folder: {adapter_dir}")
            return str(adapter_dir)

        raise FileNotFoundError(
            f"Artifact zip was extracted to {extract_dir}, but no folder containing adapter_config.json was found. "
            "Do not use paths like artifact.zip\\final_adapter on Kaggle; extract the zip first or set PARSER_ADAPTER_PATH."
        )

    raise FileNotFoundError(
        "Could not find the parser LoRA adapter. Set PARSER_ADAPTER_PATH to an extracted final_adapter folder, "
        f"or set PARSER_ARTIFACT_ZIP to a real zip file. Current zip path: {ARTIFACT_ZIP_PATH}"
    )

In [5]:
def find_config_dir() -> Path:
    candidates = []

    if "CONFIG_DIR" in globals():
        candidates.append(Path(CONFIG_DIR))

    candidates.extend([
        Path("/kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs"),
        Path("/kaggle/working/government_parser_artifact/configs"),
        Path("/kaggle/working/government_parser_artifact/final_adapter/configs"),
    ])

    for path in candidates:
        if path.exists() and path.is_dir():
            return path

    root = Path("/kaggle/input")
    if root.exists():
        for path in root.rglob("configs"):
            if path.is_dir():
                return path

    raise FileNotFoundError("Could not find parser configs directory. Set PARSER_CONFIG_DIR.")


CONFIG_DIR = find_config_dir()
print("Using CONFIG_DIR:", CONFIG_DIR)


def find_config_file(stem: str) -> Path:
    candidates = [
        CONFIG_DIR / stem,
        CONFIG_DIR / f"{stem}.json",
    ]

    for path in candidates:
        if path.exists():
            return path

    matches = list(CONFIG_DIR.glob(f"{stem}*"))
    if matches:
        return matches[0]

    raise FileNotFoundError(f"Missing config file for {stem} in {CONFIG_DIR}")


def load_config_json(stem: str) -> Any:
    path = find_config_file(stem)
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Loaded {stem}: {path.name}")
    return data


def normalize_intents(raw: Any) -> set[str]:
    if isinstance(raw, list):
        return {str(item.get("code") or item.get("intent") or item) if isinstance(item, dict) else str(item) for item in raw}

    if isinstance(raw, dict):
        for key in ["intents", "allowed_intents", "items", "data"]:
            if isinstance(raw.get(key), list):
                return {str(item.get("code") or item.get("intent") or item) if isinstance(item, dict) else str(item) for item in raw[key]}

    return set()


SCHEMA = load_config_json("parsed_query_schema.v1")
INTENTS = normalize_intents(load_config_json("parser_intents.v1"))
ENUMS = load_config_json("parser_enums.v1")
QUESTION_FAMILIES = load_config_json("question_families.v1")
COUNTRY_CATALOG = load_config_json("country_catalog.v1")
INDICATOR_CATALOG = load_config_json("indicator_catalog.v1")

FAMILY_TO_INTENT = {}
for item in QUESTION_FAMILIES.get("families", []) if isinstance(QUESTION_FAMILIES, dict) else QUESTION_FAMILIES:
    if isinstance(item, dict):
        family_id = item.get("id") or item.get("code") or item.get("question_family") or item.get("family")
        intent = item.get("intent")
        if family_id and intent:
            FAMILY_TO_INTENT[str(family_id)] = str(intent)
    elif isinstance(item, str):
        FAMILY_TO_INTENT[item] = ""

COUNTRY_CODES = {str(x.get("code")).upper() for x in COUNTRY_CATALOG.get("countries", []) if isinstance(x, dict) and x.get("code")}
INDICATOR_CODES = {str(x.get("code")) for x in INDICATOR_CATALOG.get("indicators", []) if isinstance(x, dict) and x.get("code")}
ALLOWED_GROUPS = set(ENUMS.get("country_groups", [])) if isinstance(ENUMS, dict) else set()

validator = Draft7Validator(SCHEMA)

print("Catalog loaded")
print("Intents:", len(INTENTS), "Families:", len(FAMILY_TO_INTENT))
print("Countries:", len(COUNTRY_CODES), "Indicators:", len(INDICATOR_CODES), "Groups:", len(ALLOWED_GROUPS))

Using CONFIG_DIR: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/configs
Loaded indicator_catalog.v1: indicator_catalog.v1.json
Loaded country_catalog.v1: country_catalog.v1.json
Loaded parser_enums.v1: parser_enums.v1.json
Loaded parser_intents.v1: parser_intents.v1.json
Loaded question_families.v1: question_families.v1.json
Catalog loaded
Total indicators: 56
Total countries: 88
Total intents: 27
Total question_families: 151
Intent-family mapping counts:
ANOMALY_DETECTION: 7
ANOMALY_EXPLANATION: 4
CLUSTER_BENCHMARK: 5
CLUSTER_MEMBERSHIP: 5
COMPARE_COUNTRIES: 8
COMPARE_INDICATORS: 5
COMPARE_PERIODS: 5
CONDITIONAL_FILTER: 5
COUNTRY_PROFILE: 4
COVERAGE: 5
CRISIS_ANALYSIS: 6
DATA_QUALITY: 4
FOLLOW_UP: 8
INDICATOR_DEFINITION: 7
LATEST_VALUE: 4
MISSING_DATA: 4
NEED_CLARIFICATION: 6
OFF_TOPIC: 5
RANKING: 8
RANK_BY_CHANGE: 5
RISK_ALERT: 5
THEME_SUMMARY: 5
TIME_SERIES: 6
TREND_ANALYSIS: 8
UNSUPPORTED: 5
VALUE_LOOKUP: 5
VISUALIZATION_REQUEST: 7


In [6]:
REQUIRED_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
    "chart_preference",
    "needs_clarification",
    "clarification_questions",
    "confidence",
]

DEFAULT_FALLBACK_PARSED = {
    "intent": "NEED_CLARIFICATION",
    "question_family": "parser_error",
    "indicators": [],
    "countries": [],
    "country_groups": [],
    "start_year": None,
    "end_year": None,
    "relative_time": None,
    "event_time": None,
    "ranking_order": None,
    "limit": None,
    "threshold": None,
    "aggregation": None,
    "chart_preference": "none",
    "needs_clarification": True,
    "clarification_questions": [
        "Mình chưa phân tích chắc chắn được câu hỏi. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
    ],
    "confidence": 0.0,
}

In [7]:
def build_fallback_parsed(reason: str) -> dict[str, Any]:
    parsed = dict(DEFAULT_FALLBACK_PARSED)
    parsed["clarification_questions"] = list(DEFAULT_FALLBACK_PARSED.get("clarification_questions", []))

    if reason == "invalid_json":
        parsed["question_family"] = "parser_invalid_json"
        parsed["clarification_questions"] = [
            "Mình chưa đọc được kết quả parser dưới dạng JSON hợp lệ. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    elif reason == "schema_error":
        parsed["question_family"] = "parser_schema_error"
        parsed["clarification_questions"] = [
            "Kết quả parser chưa đúng schema. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]

    return parsed

In [8]:
def extract_first_json_object(text: str) -> str | None:
    if not text:
        return None

    cleaned = text.strip()
    if cleaned.startswith("```"):
        lines = cleaned.splitlines()
        if lines and lines[0].strip().startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        cleaned = "\n".join(lines).strip()

    start = cleaned.find("{")
    if start == -1:
        return None

    depth = 0
    in_string = False
    escaped = False

    for index in range(start, len(cleaned)):
        char = cleaned[index]

        if in_string:
            if escaped:
                escaped = False
            elif char == "\\":
                escaped = True
            elif char == '"':
                in_string = False
            continue

        if char == '"':
            in_string = True
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return cleaned[start : index + 1]

    return None


def parse_model_json(raw_text: str) -> tuple[dict | None, bool, str | None]:
    json_text = extract_first_json_object(raw_text)
    if json_text is None:
        return None, False, "no_json_object_found"

    try:
        parsed = json.loads(json_text)
    except json.JSONDecodeError as exc:
        return None, False, str(exc)

    if not isinstance(parsed, dict):
        return None, False, "json_value_is_not_object"

    return parsed, True, None


def basic_schema_check(parsed: dict) -> tuple[bool, list[str]]:
    errors: list[str] = []

    missing_fields = [field for field in REQUIRED_FIELDS if field not in parsed]
    if missing_fields:
        errors.append(f"missing_required_fields: {missing_fields}")

    if "intent" in parsed and not isinstance(parsed["intent"], str):
        errors.append("intent_must_be_str")
    if "question_family" in parsed and not isinstance(parsed["question_family"], str):
        errors.append("question_family_must_be_str")

    for field in ["indicators", "countries", "country_groups", "clarification_questions"]:
        if field in parsed and not isinstance(parsed[field], list):
            errors.append(f"{field}_must_be_list")

    if "needs_clarification" in parsed and not isinstance(parsed["needs_clarification"], bool):
        errors.append("needs_clarification_must_be_bool")

    if "confidence" in parsed and parsed["confidence"] is not None and not isinstance(parsed["confidence"], (int, float)):
        errors.append("confidence_must_be_number_or_null")

    return len(errors) == 0, errors

In [9]:
# build_parser_messages is defined in the v3.1 runtime guardrail cell below.
# It uses a compact parser prompt plus catalog candidates, then finalize_decoded_output()
# performs post-generation catalog/evidence guardrails before /parse returns.

In [10]:
tokenizer = None
model = None
adapter_real_path = None
load_error = None
model_loaded = False
tokenizer_loaded = False
adapter_loaded = False


def load_parser_model():
    global tokenizer, model, adapter_real_path, load_error
    global model_loaded, tokenizer_loaded, adapter_loaded

    load_error = None
    model_loaded = False
    tokenizer_loaded = False
    adapter_loaded = False

    try:
        adapter_real_path = prepare_adapter_path()

        try:
            tokenizer = AutoTokenizer.from_pretrained(adapter_real_path, trust_remote_code=True)
            print(f"Loaded tokenizer from adapter path: {adapter_real_path}")
        except Exception as tokenizer_error:
            print(f"Could not load tokenizer from adapter path: {tokenizer_error}")
            print(f"Falling back to base tokenizer: {BASE_MODEL_ID}")
            tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer_loaded = True

        quantization_config = None
        if LOAD_IN_4BIT and torch.cuda.is_available():
            quantization_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                bnb_4bit_use_double_quant=True,
            )
        elif LOAD_IN_4BIT:
            print("4-bit loading was requested, but CUDA is not available. Loading without 4-bit quantization.")

        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_ID,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True,
        )

        model = PeftModel.from_pretrained(base_model, adapter_real_path)
        model.eval()

        adapter_loaded = True
        model_loaded = True

        print("Parser model loaded successfully.")
        return model
    except Exception as exc:
        load_error = f"{type(exc).__name__}: {exc}"
        print(f"Parser model load failed: {load_error}")
        raise


try:
    load_parser_model()
except Exception:
    print("Continuing with degraded service state. /health will expose load_error and /parse will return HTTP 503 until the model loads successfully.")

Using adapter folder from PARSER_ADAPTER_PATH: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter


Loaded tokenizer from adapter path: /kaggle/input/models/vnphongv/government-parser-qwen3-4b-lora-artifact/other/default/1/final_adapter


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


Parser model loaded successfully.


In [11]:
def generate_parser_output(
    message: str,
    context: dict | None = None,
    candidates: dict | None = None,
) -> str:
    if model is None or tokenizer is None:
        raise RuntimeError("Parser model is not loaded")

    messages = build_parser_messages(message, context, candidates)

    if hasattr(tokenizer, "apply_chat_template"):
        try:
            encoded = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            input_ids = tokenizer.apply_chat_template(
                messages,
                tokenize=True,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            encoded = {"input_ids": input_ids}
    else:
        manual_prompt = "\n".join([f"{m['role'].upper()}: {m['content']}" for m in messages])
        encoded = tokenizer(manual_prompt, return_tensors="pt")

    if isinstance(encoded, torch.Tensor):
        encoded = {"input_ids": encoded}
    else:
        encoded = dict(encoded)

    model_inputs = {}
    for key in ["input_ids", "attention_mask"]:
        value = encoded.get(key)
        if value is not None:
            model_inputs[key] = value.to(model.device)

    input_ids = model_inputs["input_ids"]

    generation_kwargs = {
        **model_inputs,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "pad_token_id": tokenizer.pad_token_id or tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    with torch.no_grad():
        output_ids = model.generate(**generation_kwargs)

    generated_ids = output_ids[0][input_ids.shape[-1]:]
    raw_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return raw_text.strip()

In [12]:
import re
import unicodedata
from typing import Any
from collections import defaultdict

INFERENCE_MODE = globals().get("INFERENCE_MODE", "v3_1_training_prompt_post_guardrails_runtime")
MAX_INPUT_TOKENS = globals().get("MAX_INPUT_TOKENS", 1024)
USE_SOFT_GROUNDING_PROMPT = globals().get("USE_SOFT_GROUNDING_PROMPT", False)


def safe_json_loads(text: str):
    text = (text or "").strip()
    start = text.find("{")
    end = text.rfind("}")
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)


def validate_parsed_query(parsed: dict):
    errors = []

    for e in sorted(validator.iter_errors(parsed), key=lambda x: x.path):
        path = ".".join(str(p) for p in e.path)
        errors.append(f"schema_error:{path}:{e.message}")

    intent = parsed.get("intent")
    family = parsed.get("question_family")

    if intent not in INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if family not in FAMILY_TO_INTENT:
        errors.append(f"invalid_question_family:{family}")
    elif intent != FAMILY_TO_INTENT[family]:
        errors.append(f"family_intent_mismatch:{family}:parsed={intent}:expected={FAMILY_TO_INTENT[family]}")

    for x in parsed.get("indicators", []) or []:
        if x not in INDICATOR_CODES:
            errors.append(f"invalid_indicator:{x}")

    for x in parsed.get("countries", []) or []:
        if x not in COUNTRY_CODES:
            errors.append(f"invalid_country:{x}")

    for x in parsed.get("country_groups", []) or []:
        if x not in ALLOWED_GROUPS:
            errors.append(f"invalid_country_group:{x}")

    for field in ["ranking_order", "chart_preference", "aggregation", "relative_time", "event_time"]:
        allowed = set(ENUMS.get(field, []))
        if parsed.get(field) not in allowed:
            errors.append(f"invalid_enum:{field}:{parsed.get(field)}")

    return errors


def schema_validate_full(parsed: dict):
    errors = []
    for e in sorted(validator.iter_errors(parsed), key=lambda x: x.path):
        path = ".".join(str(p) for p in e.path)
        errors.append(f"schema_error:{path}:{e.message}")
    return len(errors) == 0, errors


def list_micro_counts(pred, gold):
    pred = set(pred or [])
    gold = set(gold or [])
    return len(pred & gold), len(pred - gold), len(gold - pred)


def is_executable(parsed):
    if not parsed:
        return False
    if parsed.get("intent") in {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}:
        return False
    if parsed.get("needs_clarification") is True:
        return False
    return True


CRITICAL_FIELDS = [
    "intent",
    "question_family",
    "indicators",
    "countries",
    "country_groups",
    "start_year",
    "end_year",
    "relative_time",
    "event_time",
    "ranking_order",
    "limit",
    "threshold",
    "aggregation",
]


def is_dangerous_wrong(pred, gold, schema_ok):
    if not schema_ok or not is_executable(pred):
        return False

    if not is_executable(gold):
        return True

    for field in CRITICAL_FIELDS:
        if pred.get(field) != gold.get(field):
            return True

    return False


DEFAULT_FALLBACK_PARSED = {
    "intent": "NEED_CLARIFICATION",
    "question_family": "missing_indicator",
    "indicators": [],
    "countries": [],
    "country_groups": [],
    "start_year": None,
    "end_year": None,
    "relative_time": None,
    "event_time": None,
    "ranking_order": None,
    "limit": None,
    "threshold": None,
    "aggregation": None,
    "chart_preference": "none",
    "needs_clarification": True,
    "clarification_questions": [
        "Bạn muốn phân tích chỉ số nào, quốc gia nào và trong giai đoạn nào?"
    ],
    "confidence": 0.0,
}


def build_fallback_parsed(reason: str) -> dict[str, Any]:
    parsed = dict(DEFAULT_FALLBACK_PARSED)
    parsed["clarification_questions"] = list(DEFAULT_FALLBACK_PARSED["clarification_questions"])
    if reason == "invalid_json":
        parsed["clarification_questions"] = [
            "Parser chưa trả JSON hợp lệ. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    elif reason == "schema_error":
        parsed["clarification_questions"] = [
            "Parser trả kết quả chưa đúng schema. Bạn có thể nói rõ chỉ số, quốc gia và giai đoạn cần phân tích không?"
        ]
    return parsed


def normalize_text(text: str) -> str:
    text = str(text or "").lower().strip()
    text = unicodedata.normalize("NFD", text)
    text = "".join(ch for ch in text if unicodedata.category(ch) != "Mn")
    text = text.replace("_", " ")
    text = re.sub(r"[^a-z0-9%\s/.-]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def as_list(value: Any) -> list:
    if value is None:
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        return [value]
    return []


def normalize_indicator_catalog_for_grounding(raw: Any) -> dict[str, dict]:
    records = raw.get("indicators", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code") if isinstance(row, dict) else None
        if not code:
            continue

        aliases = []
        aliases.append(code)
        aliases.append(row.get("name") or code)
        aliases.extend(as_list(row.get("aliases")))

        hint = row.get("question_templates_hint") or {}
        if isinstance(hint, dict):
            aliases.extend([
                hint.get("vi"),
                hint.get("vi_no_diacritics"),
                hint.get("en"),
                hint.get("technical"),
            ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[str(code)] = {
            "code": str(code),
            "name": row.get("name") or str(code),
            "aliases": clean_aliases,
        }

    return catalog


def normalize_country_catalog_for_grounding(raw: Any) -> dict[str, dict]:
    records = raw.get("countries", []) if isinstance(raw, dict) else raw
    catalog = {}

    for row in records:
        code = row.get("code") if isinstance(row, dict) else None
        if not code:
            continue

        code = str(code).upper()
        aliases = []
        aliases.append(row.get("name") or code)
        aliases.extend(as_list(row.get("aliases")))

        hint = row.get("question_templates_hint") or {}
        if isinstance(hint, dict):
            aliases.extend([
                hint.get("vi"),
                hint.get("vi_no_diacritics"),
                hint.get("en"),
                hint.get("iso3"),
            ])

        clean_aliases = []
        seen = set()
        for alias in aliases:
            alias = str(alias or "").strip()
            if alias and alias not in seen:
                seen.add(alias)
                clean_aliases.append(alias)

        catalog[code] = {
            "code": code,
            "name": row.get("name") or code,
            "aliases": clean_aliases,
        }

    return catalog


INDICATOR_CATALOG_BY_CODE = normalize_indicator_catalog_for_grounding(INDICATOR_CATALOG)
COUNTRY_CATALOG_BY_CODE = normalize_country_catalog_for_grounding(COUNTRY_CATALOG)
ALLOWED_INTENTS = set(INTENTS)
QUESTION_FAMILY_IDS = set(FAMILY_TO_INTENT.keys())
QUESTION_FAMILY_BY_INTENT = defaultdict(list)
for family_id, intent in FAMILY_TO_INTENT.items():
    QUESTION_FAMILY_BY_INTENT[intent].append(family_id)

NON_EXECUTABLE_INTENTS = {"NEED_CLARIFICATION", "OFF_TOPIC", "UNSUPPORTED"}
EXECUTABLE_INTENTS = set(ALLOWED_INTENTS) - NON_EXECUTABLE_INTENTS

REQUIRES_INDICATOR_INTENTS = {
    "COMPARE_COUNTRIES",
    "COMPARE_INDICATORS",
    "RANKING",
    "RANK_BY_CHANGE",
    "TIME_SERIES",
    "VALUE_LOOKUP",
    "TREND_ANALYSIS",
    "ANOMALY_DETECTION",
    "VISUALIZATION",
} & ALLOWED_INTENTS

REQUIRES_TIME_INTENTS = {
    "VALUE_LOOKUP",
    "RANKING",
    "COMPARE_COUNTRIES",
    "TIME_SERIES",
    "TREND_ANALYSIS",
    "RANK_BY_CHANGE",
    "ANOMALY_DETECTION",
    "VISUALIZATION",
} & ALLOWED_INTENTS


def alias_in_text(normalized_text: str, alias: str) -> bool:
    normalized_alias = normalize_text(alias)
    if not normalized_alias:
        return False

    if len(normalized_alias) <= 3:
        return re.search(rf"(?<!\w){re.escape(normalized_alias)}(?!\w)", normalized_text) is not None

    return normalized_alias in normalized_text


def score_alias(normalized_text: str, alias: str) -> float:
    normalized_alias = normalize_text(alias)
    if not alias_in_text(normalized_text, alias):
        return 0.0

    if normalized_text == normalized_alias:
        return 1.0

    return min(0.99, 0.55 + len(normalized_alias) / 60)


def resolve_candidate_indicators(message: str, limit: int = 8) -> list[dict]:
    normalized_message = normalize_text(message)
    matches = []

    for code, meta in INDICATOR_CATALOG_BY_CODE.items():
        best_score = 0.0
        best_alias = ""

        aliases = []
        aliases.append(code)
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        for alias in aliases:
            score = score_alias(normalized_message, alias)
            if score > best_score:
                best_score = score
                best_alias = alias

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]


def resolve_candidate_countries(message: str, limit: int = 12) -> list[dict]:
    normalized_message = normalize_text(message)
    raw_message = str(message or "")
    matches = []

    raw_code_tokens = set(re.findall(r"\b[A-Z]{3}\b", raw_message))

    for code, meta in COUNTRY_CATALOG_BY_CODE.items():
        best_score = 0.0
        best_alias = ""

        if code in raw_code_tokens:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": code,
                "confidence": 1.0,
                "evidence_type": "iso3_code",
            })
            continue

        aliases = []
        aliases.append(meta.get("name") or code)
        aliases.extend(as_list(meta.get("aliases")))

        for alias in aliases:
            alias_text = str(alias or "").strip()

            if alias_text.upper() == code:
                continue

            score = score_alias(normalized_message, alias_text)
            if score > best_score:
                best_score = score
                best_alias = alias_text

        if best_score >= 0.6:
            matches.append({
                "code": code,
                "name": meta.get("name") or code,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
                "evidence_type": "alias",
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]


def resolve_candidate_country_groups(message: str, limit: int = 8) -> list[dict]:
    normalized_message = normalize_text(message)
    raw_message = str(message or "")
    raw_group_tokens = set(re.findall(r"\b[A-Z0-9_]{2,30}\b", raw_message.upper()))
    matches = []

    for group in sorted(ALLOWED_GROUPS):
        group_text = str(group or "").strip()
        group_upper = group_text.upper()
        best_score = 0.0
        best_alias = ""
        evidence_type = "alias"

        if group_upper in raw_group_tokens:
            best_score = 1.0
            best_alias = group_text
            evidence_type = "group_code"
        else:
            aliases = [group_text, group_text.replace("_", " ")]
            for alias in aliases:
                score = score_alias(normalized_message, alias)
                if score > best_score:
                    best_score = score
                    best_alias = alias

        if best_score >= 0.6:
            matches.append({
                "code": group,
                "matched_alias": best_alias,
                "confidence": round(best_score, 3),
                "evidence_type": evidence_type,
            })

    matches.sort(key=lambda item: item["confidence"], reverse=True)

    result = []
    seen = set()
    for item in matches:
        if item["code"] not in seen:
            seen.add(item["code"])
            result.append(item)

    return result[:limit]


def resolve_candidate_years(message: str) -> list[int]:
    years = []
    for match in re.finditer(r"\b(19\d{2}|20\d{2}|21\d{2})\b", str(message or "")):
        year = int(match.group(1))
        if 1900 <= year <= 2199 and year not in years:
            years.append(year)
    return years


def add_families_for_intent(families: list[str], intent: str):
    for family in QUESTION_FAMILY_BY_INTENT.get(intent, []):
        if family not in families:
            families.append(family)


def infer_candidate_intents_and_families(message: str, indicators: list, countries: list, country_groups: list, years: list) -> tuple[list[str], list[str]]:
    normalized = normalize_text(message)
    intents = []
    families = []

    def add_intent(intent: str):
        if intent in ALLOWED_INTENTS and intent not in intents:
            intents.append(intent)
            add_families_for_intent(families, intent)

    if any(token in normalized for token in ["viet sql", "write sql", "raw sql", "cau lenh sql", "lệnh sql"]):
        add_intent("UNSUPPORTED")
    elif any(token in normalized for token in ["du bao", "forecast", "arima", "predict", "prediction", "mo hinh", "mô hình"]):
        add_intent("UNSUPPORTED")
    elif any(token in normalized for token in ["so sanh", "compare", "doi chieu", "đối chiếu", "voi", "với"]):
        add_intent("COMPARE_COUNTRIES")
    elif any(token in normalized for token in ["top", "cao nhat", "cao nhất", "thap nhat", "thấp nhất", "xep hang", "xếp hạng", "ranking", "nuoc nao", "nước nào"]):
        add_intent("RANKING")
    elif any(token in normalized for token in ["xu huong", "xu hướng", "trend", "dien bien", "diễn biến"]):
        add_intent("TREND_ANALYSIS")
        add_intent("TIME_SERIES")
    elif any(token in normalized for token in ["ve", "vẽ", "chart", "bieu do", "biểu đồ", "line chart", "bar chart"]):
        add_intent("VISUALIZATION")
        add_intent("TIME_SERIES")
    elif any(token in normalized for token in ["du lieu", "dữ liệu", "coverage", "bao phu", "bao phủ", "thieu du lieu", "thiếu dữ liệu"]):
        add_intent("COVERAGE")
    elif indicators and (countries or country_groups) and years:
        add_intent("VALUE_LOOKUP")
    elif indicators and (countries or country_groups):
        add_intent("NEED_CLARIFICATION")
    elif countries or country_groups:
        add_intent("NEED_CLARIFICATION")

    if not intents:
        add_intent("NEED_CLARIFICATION")

    return intents, families[:16]


def build_grounding_candidates(message: str, context: dict | None = None) -> dict[str, Any]:
    candidate_indicators = resolve_candidate_indicators(message)
    candidate_countries = resolve_candidate_countries(message)
    candidate_country_groups = resolve_candidate_country_groups(message)
    detected_years = resolve_candidate_years(message)
    candidate_intents, candidate_question_families = infer_candidate_intents_and_families(
        message,
        candidate_indicators,
        candidate_countries,
        candidate_country_groups,
        detected_years,
    )

    return {
        "candidate_indicators": candidate_indicators,
        "candidate_countries": candidate_countries,
        "candidate_country_groups": candidate_country_groups,
        "detected_years": detected_years,
        "candidate_intents": candidate_intents,
        "candidate_question_families": candidate_question_families,
    }


def compact_candidates_for_prompt(candidates: dict[str, Any]) -> dict[str, Any]:
    return {
        "candidate_indicators": [
            {
                "code": item.get("code"),
                "name": item.get("name"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_indicators", [])
        ],
        "candidate_countries": [
            {
                "code": item.get("code"),
                "name": item.get("name"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_countries", [])
        ],
        "candidate_country_groups": [
            {
                "code": item.get("code"),
                "matched_alias": item.get("matched_alias"),
            }
            for item in candidates.get("candidate_country_groups", [])
        ],
        "detected_years": candidates.get("detected_years", []),
        "candidate_intents": candidates.get("candidate_intents", []),
        "candidate_question_families": candidates.get("candidate_question_families", []),
    }


def build_parser_messages(message: str, context: dict | None = None, candidates: dict | None = None) -> list[dict[str, str]]:
    candidates = candidates or build_grounding_candidates(message, context)

    system_prompt = """
You are a semantic parser for a Government Economic Indicator AI Agent.
Return only one valid JSON object matching ParsedQuery v1.
Do not answer the question.
Do not write SQL.
Do not query data.
Do not invent missing indicators, countries, country groups, years, or question families.
Prefer NEED_CLARIFICATION over guessing.
Only use valid intent, question_family, indicator code, country code, and enum values from the provided catalogs/candidates.
If the user asks for forecasting, ARIMA/modeling, raw SQL, causal claims, or unsupported operations, return UNSUPPORTED.
""".strip()

    user_payload = {
        "message": message,
        "context": context,
        "grounding_candidates": compact_candidates_for_prompt(candidates),
        "required_output_fields": [
            "intent",
            "question_family",
            "indicators",
            "countries",
            "country_groups",
            "start_year",
            "end_year",
            "relative_time",
            "event_time",
            "ranking_order",
            "limit",
            "threshold",
            "aggregation",
            "chart_preference",
            "needs_clarification",
            "clarification_questions",
            "confidence",
        ],
    }

    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": json.dumps(user_payload, ensure_ascii=False)},
    ]


def build_parser_messages_for_row(row: dict[str, Any], user_message: str, candidates: dict | None = None) -> list[dict[str, str]]:
    """Build the prompt for v3 evaluation.

    Default mode keeps the original training/evaluation prompt format by reusing
    the system+user messages from the dataset. Catalog grounding is still built
    and used by post-generation guardrails, but it is not injected into the
    prompt unless USE_SOFT_GROUNDING_PROMPT=True.
    """
    if USE_SOFT_GROUNDING_PROMPT:
        return build_parser_messages(user_message, context=None, candidates=candidates)

    messages = row.get("messages") or []
    if len(messages) >= 2:
        prompt_messages = []
        for message_item in messages[:2]:
            role = str(message_item.get("role") or "").strip()
            content = str(message_item.get("content") or "")
            if role and content:
                prompt_messages.append({"role": role, "content": content})
        if prompt_messages:
            return prompt_messages

    return [
        {
            "role": "system",
            "content": "You are a semantic parser for a Government Economic Indicator AI Agent. Output only one valid JSON object. Do not answer the question.",
        },
        {"role": "user", "content": user_message},
    ]


def safe_list(value: Any) -> list:
    if isinstance(value, list):
        return value
    if value is None:
        return []
    return [value]


def candidate_codes(candidates: dict | None, key: str) -> set[str]:
    values = set()
    items = (candidates or {}).get(key) or []

    for item in items:
        if isinstance(item, dict):
            code = item.get("code")
            if code:
                values.add(str(code))
        elif item is not None:
            values.add(str(item))

    return values


def detected_year_set(candidates: dict | None) -> set[int]:
    result = set()
    for item in (candidates or {}).get("detected_years") or []:
        try:
            result.add(int(item))
        except Exception:
            pass
    return result


def normalize_parsed_basic(parsed: dict) -> dict:
    normalized = dict(DEFAULT_FALLBACK_PARSED)
    normalized.update(parsed or {})

    normalized["intent"] = str(normalized.get("intent") or "").strip()
    normalized["question_family"] = str(normalized.get("question_family") or "").strip()

    normalized["indicators"] = [
        str(item).strip()
        for item in safe_list(normalized.get("indicators"))
        if item is not None and str(item).strip()
    ]

    normalized["countries"] = [
        str(item).strip().upper()
        for item in safe_list(normalized.get("countries"))
        if item is not None and str(item).strip()
    ]

    normalized["country_groups"] = [
        str(item).strip()
        for item in safe_list(normalized.get("country_groups"))
        if item is not None and str(item).strip()
    ]

    normalized["clarification_questions"] = [
        str(item).strip()
        for item in safe_list(normalized.get("clarification_questions"))
        if item is not None and str(item).strip()
    ]

    for field in ["start_year", "end_year", "limit"]:
        value = normalized.get(field)
        if value is not None:
            try:
                normalized[field] = int(value)
            except Exception:
                pass

    value = normalized.get("confidence")
    try:
        normalized["confidence"] = float(value)
    except Exception:
        normalized["confidence"] = 0.0

    return normalized


def family_allowed_for_intent(question_family: str, intent: str) -> bool:
    if not question_family or not intent:
        return False

    if question_family not in QUESTION_FAMILY_IDS:
        return False

    expected_intent = FAMILY_TO_INTENT.get(question_family)
    if expected_intent:
        return expected_intent == intent

    if intent == "NEED_CLARIFICATION":
        return question_family.startswith("missing_") or question_family.startswith("ambiguous_")

    if intent == "UNSUPPORTED":
        return question_family.startswith("unsupported_")

    return True


def derive_safe_question_family(parsed: dict, candidates: dict | None) -> str | None:
    intent = parsed.get("intent")

    def allowed(family: str) -> str | None:
        if family in QUESTION_FAMILY_IDS and family_allowed_for_intent(family, intent):
            return family
        return None

    if intent == "COMPARE_COUNTRIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")

        if len(countries) >= 2:
            if start_year is not None and end_year is not None and start_year == end_year:
                return allowed("compare_countries_single_year")
            if start_year is not None and end_year is not None:
                return allowed("compare_countries_period")

    if intent == "RANKING":
        order = parsed.get("ranking_order")
        if order == "desc":
            return allowed("ranking_top_n")
        if order == "asc":
            return allowed("ranking_bottom_n")

    if intent == "TREND_ANALYSIS":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("trend_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("trend_multi_country_period")

    if intent == "TIME_SERIES":
        countries = parsed.get("countries") or []
        start_year = parsed.get("start_year")
        end_year = parsed.get("end_year")
        if len(countries) == 1 and start_year is not None and end_year is not None:
            return allowed("time_series_single_country_period")
        if len(countries) > 1 and start_year is not None and end_year is not None:
            return allowed("time_series_multi_country_period")

    return None


def make_clarification(parsed: dict, question: str, family: str) -> dict:
    updated = dict(parsed)
    updated["intent"] = "NEED_CLARIFICATION"
    updated["question_family"] = family
    updated["needs_clarification"] = True
    updated["clarification_questions"] = [question]
    updated["confidence"] = min(float(updated.get("confidence") or 0.6), 0.6)
    return updated


def infer_intent_from_question_family(question_family: str | None) -> str | None:
    if question_family in FAMILY_TO_INTENT:
        return FAMILY_TO_INTENT[question_family]
    return None


def normalize_intent_and_family_for_guard(parsed: dict, repairs_applied: list[str]) -> dict:
    normalized = dict(parsed)
    intent = normalized.get("intent")
    family = normalized.get("question_family")

    if intent not in ALLOWED_INTENTS and intent in FAMILY_TO_INTENT:
        if family not in QUESTION_FAMILY_IDS:
            normalized["question_family"] = intent
            repairs_applied.append(f"question_family:{family}->{intent}")
        new_intent = FAMILY_TO_INTENT[intent]
        repairs_applied.append(f"intent:{intent}->{new_intent}")
        normalized["intent"] = new_intent
        return normalized

    if normalized.get("intent") not in ALLOWED_INTENTS:
        derived_intent = infer_intent_from_question_family(family)
        if derived_intent:
            repairs_applied.append(f"intent:{intent}->{derived_intent}")
            normalized["intent"] = derived_intent

    intent = normalized.get("intent")
    family = normalized.get("question_family")

    if family not in QUESTION_FAMILY_IDS:
        derived_family = derive_safe_question_family(normalized, None)
        if derived_family:
            repairs_applied.append(f"question_family:{family}->{derived_family}")
            normalized["question_family"] = derived_family

    intent = normalized.get("intent")
    family = normalized.get("question_family")
    expected_intent = FAMILY_TO_INTENT.get(family)

    if intent in ALLOWED_INTENTS and family in QUESTION_FAMILY_IDS and expected_intent != intent:
        derived_family = derive_safe_question_family(normalized, None)
        if derived_family and FAMILY_TO_INTENT.get(derived_family) == intent:
            repairs_applied.append(f"question_family:{family}->{derived_family}")
            normalized["question_family"] = derived_family
        elif expected_intent:
            repairs_applied.append(f"intent:{intent}->{expected_intent}")
            normalized["intent"] = expected_intent

    return normalized


def text_has_value_evidence(message: str, value: Any) -> bool:
    if value is None:
        return False
    normalized_message = normalize_text(message)
    normalized_value = normalize_text(value)
    if not normalized_value:
        return False
    if normalized_value in {"after_event", "before_event", "during_event", "recent", "latest"}:
        return False
    return normalized_value in normalized_message


def has_time_evidence(parsed: dict, message: str, candidates: dict | None = None) -> bool:
    if detected_year_set(candidates):
        return True

    relative_time = parsed.get("relative_time")
    event_time = parsed.get("event_time")

    if event_time is not None and text_has_value_evidence(message, event_time):
        return True

    normalized_message = normalize_text(message)
    relative_cues = [
        "gan day", "moi nhat", "hien nay", "sau", "truoc", "trong", "during", "after", "before", "recent", "latest"
    ]
    if relative_time is not None and any(cue in normalized_message for cue in relative_cues):
        return True

    return False


def validate_catalog_and_guard(parsed: dict, message: str, candidates: dict | None = None) -> dict:
    errors = []
    repairs_applied = []

    normalized = normalize_parsed_basic(parsed)
    normalized = normalize_intent_and_family_for_guard(normalized, repairs_applied)

    intent = normalized.get("intent")
    question_family = normalized.get("question_family")

    candidate_indicator_codes = candidate_codes(candidates, "candidate_indicators")
    candidate_country_codes = candidate_codes(candidates, "candidate_countries")
    candidate_group_codes = candidate_codes(candidates, "candidate_country_groups")
    years_in_user = detected_year_set(candidates)

    if intent not in ALLOWED_INTENTS:
        errors.append(f"invalid_intent:{intent}")

    if question_family not in QUESTION_FAMILY_IDS:
        errors.append(f"invalid_question_family:{question_family}")
    elif intent in ALLOWED_INTENTS and FAMILY_TO_INTENT.get(question_family) != intent:
        errors.append(f"family_intent_mismatch:{question_family}:parsed={intent}:expected={FAMILY_TO_INTENT.get(question_family)}")

    for indicator in normalized.get("indicators", []):
        if indicator not in INDICATOR_CATALOG_BY_CODE:
            errors.append(f"invalid_indicator:{indicator}")
        elif candidate_indicator_codes and indicator not in candidate_indicator_codes:
            errors.append(f"indicator_not_mentioned_by_user:{indicator}")
        elif not candidate_indicator_codes and intent in REQUIRES_INDICATOR_INTENTS:
            errors.append(f"indicator_without_user_evidence:{indicator}")

    for country in normalized.get("countries", []):
        if country not in COUNTRY_CATALOG_BY_CODE:
            errors.append(f"invalid_country:{country}")
        elif candidate_country_codes and country not in candidate_country_codes:
            errors.append(f"country_not_mentioned_by_user:{country}")
        elif not candidate_country_codes and not candidate_group_codes:
            errors.append(f"country_without_user_evidence:{country}")

    for group in normalized.get("country_groups", []):
        if group not in ALLOWED_GROUPS:
            errors.append(f"invalid_country_group:{group}")
        elif candidate_group_codes and group not in candidate_group_codes:
            errors.append(f"country_group_not_mentioned_by_user:{group}")
        elif not candidate_group_codes:
            errors.append(f"country_group_without_user_evidence:{group}")

    if intent == "UNSUPPORTED":
        normalized["needs_clarification"] = False
        normalized["clarification_questions"] = []
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "unsupported_request" if len(errors) == 0 else "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent == "OFF_TOPIC":
        normalized["needs_clarification"] = False
        normalized["clarification_questions"] = []
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "off_topic" if len(errors) == 0 else "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent == "NEED_CLARIFICATION" or normalized.get("needs_clarification") is True:
        normalized["intent"] = "NEED_CLARIFICATION"
        normalized["needs_clarification"] = True
        if not normalized.get("clarification_questions"):
            normalized["clarification_questions"] = [
                "Bạn muốn phân tích chỉ số nào, quốc gia nào và trong giai đoạn nào?"
            ]
        return {
            "parsed": normalized,
            "catalog_pass": len(errors) == 0,
            "safe_to_execute": False,
            "fallback_reason": "needs_clarification" if len(errors) == 0 else "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent in REQUIRES_INDICATOR_INTENTS:
        if not normalized.get("indicators") or not candidate_indicator_codes:
            fallback = make_clarification(
                normalized,
                "Bạn muốn so sánh/phân tích chỉ số nào?",
                "missing_indicator",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_indicator_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

    if intent in REQUIRES_TIME_INTENTS:
        start_year = normalized.get("start_year")
        end_year = normalized.get("end_year")

        if start_year is None or end_year is None or not has_time_evidence(normalized, message, candidates):
            fallback = make_clarification(
                normalized,
                "Bạn muốn phân tích trong năm nào hoặc giai đoạn nào?",
                "ambiguous_time_range",
            )
            return {
                "parsed": fallback,
                "catalog_pass": len(errors) == 0,
                "safe_to_execute": False,
                "fallback_reason": "missing_time_in_user_message",
                "repairs_applied": repairs_applied,
                "validation_errors": errors,
            }

        if years_in_user:
            if isinstance(start_year, int) and start_year not in years_in_user:
                errors.append(f"start_year_not_mentioned_by_user:{start_year}")
            if isinstance(end_year, int) and end_year not in years_in_user:
                errors.append(f"end_year_not_mentioned_by_user:{end_year}")

    if errors:
        fallback = make_clarification(
            normalized,
            "Mình chưa xác thực được chỉ số, quốc gia hoặc thời gian trong câu hỏi. Bạn có thể nói rõ hơn không?",
            "ambiguous_indicator",
        )
        return {
            "parsed": fallback,
            "catalog_pass": False,
            "safe_to_execute": False,
            "fallback_reason": "catalog_validation_failed",
            "repairs_applied": repairs_applied,
            "validation_errors": errors,
        }

    if intent not in EXECUTABLE_INTENTS:
        return {
            "parsed": normalized,
            "catalog_pass": True,
            "safe_to_execute": False,
            "fallback_reason": "non_executable_intent",
            "repairs_applied": repairs_applied,
            "validation_errors": [],
        }

    return {
        "parsed": normalized,
        "catalog_pass": True,
        "safe_to_execute": True,
        "fallback_reason": None,
        "repairs_applied": repairs_applied,
        "validation_errors": [],
    }


def finalize_decoded_output(decoded: str, message: str, candidates: dict[str, Any]) -> dict[str, Any]:
    raw_pred = None
    eval_pred = None
    deployment_pred = None
    json_ok = False
    raw_schema_ok = False
    raw_validation_errors = []
    catalog_pass = False
    safe_to_execute = False
    fallback_reason = None
    repairs_applied = []
    guard_errors = []
    final_validation_errors = []

    try:
        raw_pred = safe_json_loads(decoded)
        json_ok = True
    except Exception as exc:
        raw_validation_errors.append(f"json_error:{str(exc)}")
        deployment_pred = build_fallback_parsed("invalid_json")
        fallback_reason = "invalid_json"

    if raw_pred is not None:
        raw_validation_errors = validate_parsed_query(raw_pred)
        raw_schema_ok = len(raw_validation_errors) == 0
        eval_pred = raw_pred

        guard_result = validate_catalog_and_guard(raw_pred, message, candidates)
        deployment_pred = guard_result["parsed"]
        catalog_pass = guard_result["catalog_pass"]
        safe_to_execute = guard_result["safe_to_execute"]
        fallback_reason = guard_result["fallback_reason"]
        repairs_applied = guard_result["repairs_applied"]
        guard_errors.extend(guard_result["validation_errors"])

        deployment_schema_pass, deployment_schema_errors = schema_validate_full(deployment_pred)
        final_validation_errors.extend(deployment_schema_errors)

        if not deployment_schema_pass:
            catalog_pass = False
            safe_to_execute = False
            fallback_reason = "schema_error_after_guard"
            guard_errors.extend(deployment_schema_errors)
            deployment_pred = build_fallback_parsed("schema_error")
            final_validation_errors = validate_parsed_query(deployment_pred)

    if eval_pred is None:
        eval_pred = deployment_pred

    if deployment_pred is not None and not final_validation_errors:
        final_validation_errors = validate_parsed_query(deployment_pred)

    return {
        "raw_pred": raw_pred,
        "pred": eval_pred,
        "deployment_pred": deployment_pred,
        "json_ok": json_ok,
        "raw_schema_ok": raw_schema_ok,
        "schema_ok": raw_schema_ok,
        "catalog_pass": catalog_pass,
        "safe_to_execute": safe_to_execute,
        "fallback_reason": fallback_reason,
        "repairs_applied": repairs_applied,
        "errors": raw_validation_errors,
        "guard_errors": guard_errors,
        "final_validation_errors": final_validation_errors,
    }

print("Inference mode:", INFERENCE_MODE, flush=True)
print("MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS, flush=True)
print("USE_SOFT_GROUNDING_PROMPT:", USE_SOFT_GROUNDING_PROMPT, flush=True)
print("Grounding catalogs:", len(INDICATOR_CATALOG_BY_CODE), "indicators,", len(COUNTRY_CATALOG_BY_CODE), "countries", flush=True)


Loaded parsed_query_schema.v1: parsed_query_schema.v1.json


In [13]:
class ParseRequest(BaseModel):
    message: str
    context: dict[str, Any] | None = None
    debug: bool = False


class ParseResponse(BaseModel):
    parsed: dict[str, Any]
    raw_model_output: str | None = None
    raw_parsed: dict[str, Any] | None = None
    deployment_parsed: dict[str, Any] | None = None
    valid_json: bool
    raw_schema_pass: bool
    deployment_schema_pass: bool
    schema_pass: bool
    catalog_pass: bool
    safe_to_execute: bool
    fallback_reason: str | None = None
    repairs_applied: list[str] = Field(default_factory=list)
    candidates: dict[str, Any] = Field(default_factory=dict)
    validation_errors: list[str] = Field(default_factory=list)
    raw_validation_errors: list[str] = Field(default_factory=list)
    guard_errors: list[str] = Field(default_factory=list)
    final_validation_errors: list[str] = Field(default_factory=list)
    inference_mode: str = INFERENCE_MODE
    latency_ms: int


app = FastAPI(title="Government Parser Model Service", version=SERVICE_VERSION)


@app.get("/health")
def health() -> dict[str, Any]:
    if model_loaded and tokenizer_loaded and adapter_loaded:
        status = "ok"
    elif load_error:
        status = "error"
    else:
        status = "degraded"

    return {
        "status": status,
        "service": "parser-model-service",
        "version": SERVICE_VERSION,
        "inference_mode": INFERENCE_MODE,
        "model_loaded": model_loaded,
        "tokenizer_loaded": tokenizer_loaded,
        "adapter_loaded": adapter_loaded,
        "base_model_id": BASE_MODEL_ID,
        "adapter_path": adapter_real_path,
        "load_error": load_error,
    }


@app.post("/parse", response_model=ParseResponse)
def parse(req: ParseRequest):
    start = time.time()

    message = (req.message or "").strip()
    if not message:
        raise HTTPException(status_code=400, detail="message is required")

    if not model_loaded or model is None or tokenizer is None:
        raise HTTPException(
            status_code=503,
            detail={
                "message": "Parser model is not loaded",
                "load_error": load_error,
            },
        )

    candidates = build_grounding_candidates(message, req.context)
    raw_output = generate_parser_output(message=message, context=req.context, candidates=candidates)
    result = finalize_decoded_output(raw_output, message, candidates)

    deployment_parsed = result.get("deployment_pred") or build_fallback_parsed("invalid_json")
    final_validation_errors = result.get("final_validation_errors") or []
    guard_errors = result.get("guard_errors") or []
    raw_validation_errors = result.get("errors") or []
    deployment_schema_pass = len(final_validation_errors) == 0

    validation_errors = []
    validation_errors.extend(raw_validation_errors)
    validation_errors.extend(guard_errors)
    validation_errors.extend(final_validation_errors)

    latency_ms = int((time.time() - start) * 1000)

    return ParseResponse(
        parsed=deployment_parsed,
        raw_model_output=raw_output if req.debug else None,
        raw_parsed=result.get("raw_pred") if req.debug else None,
        deployment_parsed=deployment_parsed if req.debug else None,
        valid_json=bool(result.get("json_ok")),
        raw_schema_pass=bool(result.get("raw_schema_ok")),
        deployment_schema_pass=deployment_schema_pass,
        schema_pass=deployment_schema_pass,
        catalog_pass=bool(result.get("catalog_pass")),
        safe_to_execute=bool(result.get("safe_to_execute")) and deployment_schema_pass,
        fallback_reason=result.get("fallback_reason"),
        repairs_applied=result.get("repairs_applied") or [],
        candidates=candidates,
        validation_errors=validation_errors,
        raw_validation_errors=raw_validation_errors,
        guard_errors=guard_errors,
        final_validation_errors=final_validation_errors,
        inference_mode=INFERENCE_MODE,
        latency_ms=latency_ms,
    )

In [14]:
test_cases = [
    "So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023",
    "Top 10 nước có lạm phát CPI cao nhất năm 2020",
    "Xu hướng thất nghiệp của Việt Nam từ 2000 đến 2022",
    "Xu hướng thất nghiệp của Nhật Bản sau COVID",
    "Việt Nam có dữ liệu nợ công từ năm nào đến năm nào?",
    "So sánh Việt Nam với Thái Lan",
    "Dự báo GDP Việt Nam 2030 bằng mô hình ARIMA",
]

for message in test_cases:
    print("=" * 100)
    print("MESSAGE:", message)

    candidates = build_grounding_candidates(message)
    print("CANDIDATES:")
    print(json.dumps(candidates, ensure_ascii=False, indent=2))

    if not model_loaded:
        print("SKIP model inference because model is not loaded.")
        print("load_error:", load_error)
        continue

    raw = generate_parser_output(message, None, candidates)
    result = finalize_decoded_output(raw, message, candidates)

    print("RAW:")
    print(raw)
    print("VALID_JSON:", result["json_ok"])
    print("RAW_SCHEMA_PASS:", result["raw_schema_ok"])
    print("DEPLOYMENT_SCHEMA_PASS:", len(result.get("final_validation_errors") or []) == 0)
    print("CATALOG_PASS:", result["catalog_pass"])
    print("SAFE_TO_EXECUTE:", result["safe_to_execute"])
    print("FALLBACK_REASON:", result["fallback_reason"])
    print("REPAIRS_APPLIED:", result["repairs_applied"])
    print("RAW_VALIDATION_ERRORS:", result["errors"])
    print("GUARD_ERRORS:", result.get("guard_errors", []))
    print("FINAL_VALIDATION_ERRORS:", result.get("final_validation_errors", []))
    print("RAW_PARSED:")
    print(json.dumps(result.get("raw_pred"), ensure_ascii=False, indent=2))
    print("DEPLOYMENT_PARSED:")
    print(json.dumps(result.get("deployment_pred"), ensure_ascii=False, indent=2))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


MESSAGE: So sánh nợ công Việt Nam và Thái Lan từ 2010 đến 2023
CANDIDATES:
{
  "candidate_indicators": [
    {
      "code": "govdebt_GDP",
      "name": "Government Debt / GDP",
      "matched_alias": "nợ công",
      "confidence": 0.667
    }
  ],
  "candidate_countries": [
    {
      "code": "THA",
      "name": "Thailand",
      "matched_alias": "Thái Lan",
      "confidence": 0.683
    },
    {
      "code": "VNM",
      "name": "Vietnam",
      "matched_alias": "Viet Nam",
      "confidence": 0.683
    }
  ],
  "detected_years": [
    2010,
    2023
  ],
  "candidate_intents": [
    "COMPARE_COUNTRIES"
  ],
  "candidate_question_families": [
    "compare_countries_period",
    "compare_countries_single_year",
    "compare_countries_after_event",
    "compare_countries_before_after_event",
    "compare_country_vs_group_average",
    "compare_country_vs_region",
    "compare_country_vs_peer",
    "compare_countries_volatility"
  ]
}
RAW:
{"intent":"COMPARE_COUNTRIES","question_fam

In [15]:
def run_api():
    uvicorn.run(app, host="0.0.0.0", port=PORT)


server_thread = threading.Thread(target=run_api, daemon=True)
server_thread.start()

print(f"FastAPI server running on port {PORT}")

FastAPI server running on port 8000


In [16]:
if not NGROK_AUTHTOKEN:
    print("WARNING: NGROK_AUTHTOKEN is not set. Skip ngrok tunnel.")
    public_url = None
else:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    tunnel = ngrok.connect(PORT, "http")
    public_url = tunnel.public_url
    print("Public URL:", public_url)
    print(f"GET  {public_url}/health")
    print(f"POST {public_url}/parse")

INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Public URL: NgrokTunnel: "https://kadence-unsulphonated-carlene.ngrok-free.dev" -> "http://localhost:8000"
GET  NgrokTunnel: "https://kadence-unsulphonated-carlene.ngrok-free.dev" -> "http://localhost:8000"/health
POST NgrokTunnel: "https://kadence-unsulphonated-carlene.ngrok-free.dev" -> "http://localhost:8000"/parse
